In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from datetime import datetime
import os

def load_idx_images(filepath):
    import struct
    with open(filepath, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)
    return images

def load_idx_labels(filepath):
    import struct
    with open(filepath, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

def prepare_data(images, labels, num_classes=62):
    X = images.astype('float32') / 255.0
    X = np.expand_dims(X, axis=-1)

    y = keras.utils.to_categorical(labels, num_classes)

    return X, y

def create_simple_model(num_classes=62, input_shape=(28, 28, 1)):
    model = keras.Sequential([
        # Conv Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Conv Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Fully Connected
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

def create_model(num_classes=62, input_shape=(28, 28, 1)):
    model = keras.Sequential([
        # Conv Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Conv Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Conv Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Fully Connected
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model

def get_callbacks(model_name, initial_lr=0.001):
    checkpoint = keras.callbacks.ModelCheckpoint(
        f'{model_name}_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )

    def lr_schedule(epoch):
        if epoch < 15:
            return initial_lr
        elif epoch < 30:
            return initial_lr * 0.5
        elif epoch < 45:
            return initial_lr * 0.1
        else:
            return initial_lr * 0.01

    lr_scheduler = keras.callbacks.LearningRateScheduler(lr_schedule, verbose=1)

    return [checkpoint, early_stop, reduce_lr, lr_scheduler]

def plot_history(history, dataset_name):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'{dataset_name} - Accuracy', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{dataset_name} - Loss', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{dataset_name}_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

from sklearn.metrics import f1_score, classification_report
import numpy as np
from datetime import datetime

def evaluate_model(model, X_test, y_test, dataset_name, save_report=True):
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    y_prob = model.predict(X_test, batch_size=256, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    print(f"\n{'='*70}")
    print(f"{dataset_name} - FINAL EVALUATION")
    print(f"{'='*70}")
    print(f"Test Loss      : {test_loss:.4f}")
    print(f"Test Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"F1-score Macro : {f1_macro:.4f}")
    print(f"F1-score Weight: {f1_weighted:.4f}")
    print(f"{'='*70}\n")

    with open("training_results.txt", "a") as f:
        f.write(f"{dataset_name} - Results\n")
        f.write(f"Test Loss      : {test_loss:.4f}\n")
        f.write(f"Test Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)\n")
        f.write(f"F1-score Macro : {f1_macro:.4f}\n")
        f.write(f"F1-score Weight: {f1_weighted:.4f}\n")
        f.write(f"Timestamp      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    if save_report:
        report = classification_report(y_true, y_pred, digits=4)
        print(report)
        with open(f"{dataset_name}_classification_report.txt", "w") as f:
            f.write(report)

    return {
        "loss": test_loss,
        "accuracy": test_acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

def evaluate_model_generator(model, test_generator, dataset_name, save_report=True):
    test_loss, test_acc = model.evaluate(test_generator, verbose=1)

    print("Collecting predictions...")
    y_true = []
    y_pred = []

    for i in range(len(test_generator)):
        X_batch, y_batch = test_generator[i]
        pred_batch = model.predict(X_batch, verbose=0)

        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(pred_batch, axis=1))

        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(test_generator)} batches")

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Calculate F1 scores
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    # Print results
    print(f"\n{'='*70}")
    print(f"{dataset_name} - FINAL EVALUATION")
    print(f"{'='*70}")
    print(f"Test Loss      : {test_loss:.4f}")
    print(f"Test Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"F1-score Macro : {f1_macro:.4f}")
    print(f"F1-score Weight: {f1_weighted:.4f}")
    print(f"{'='*70}\n")

    # Save to file
    with open("training_results.txt", "a") as f:
        f.write(f"\n{dataset_name} - Results\n")
        f.write(f"Test Loss      : {test_loss:.4f}\n")
        f.write(f"Test Accuracy  : {test_acc:.4f} ({test_acc*100:.2f}%)\n")
        f.write(f"F1-score Macro : {f1_macro:.4f}\n")
        f.write(f"F1-score Weight: {f1_weighted:.4f}\n")
        f.write(f"Timestamp      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"{'-'*70}\n")

    # Save classification report
    if save_report:
        report = classification_report(y_true, y_pred, digits=4)
        print(report)

        report_filename = f"{dataset_name.replace(' ', '_').replace('-', '_')}_classification_report.txt"
        with open(report_filename, "w") as f:
            f.write(f"{dataset_name} - Classification Report\n")
            f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"{'='*70}\n\n")
            f.write(report)
        print(f"\nClassification report saved to: {report_filename}")

    return {
        "loss": test_loss,
        "accuracy": test_acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

upload_dir = "/content/drive/MyDrive/data"

os.makedirs(upload_dir, exist_ok=True)

print("Upload directory ready:", upload_dir)

In [ ]:
from google.colab import drive
import zipfile, os, shutil

drive.mount('/content/drive')

zips = {
    "original":  "/content/drive/MyDrive/data/original.zip",
    "balanced":  "/content/drive/MyDrive/data/balanced.zip",
    "augmented":  "/content/drive/MyDrive/data/augmented.zip",
    "combined":  "/content/drive/MyDrive/data/combined.zip",
}

base_dir = "/content/data"
os.makedirs(base_dir, exist_ok=True)

for name, zip_path in zips.items():
    print(f"\nExtracting {name}.zip")

    tmp = f"/content/tmp"
    out = f"{base_dir}/{name}"

    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    os.makedirs(tmp)

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(tmp)

    files = os.listdir(tmp)
    src = os.path.join(tmp, files[0]) if len(files) == 1 else tmp

    if os.path.exists(out):
        shutil.rmtree(out)
    shutil.move(src, out)

    shutil.rmtree(tmp)
    print(f"Done -> {out}")

In [ ]:
print("TRAINING ON ORIGINAL DATASET - SIMPLE MODEL")

# Load data
def rotate(images):
    images = np.array([np.rot90(np.fliplr(img)) for img in images])
    return images

train_images = load_idx_images('data/original/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('data/original/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('data/original/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('data/original/emnist-byclass-test-labels-idx1-ubyte')

train_images = rotate(train_images)
test_images = rotate(test_images)

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile simple model
simple_model_original = create_simple_model()
simple_model_original.summary()

initial_lr = 0.001
optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
simple_model_original.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_simple_original = simple_model_original.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_original_simple_model'),
    verbose=1
)

# Plot history
plot_history(history_simple_original, 'Original Dataset - Simple Model')

# Evaluate
results_simple_orig = evaluate_model(
    simple_model_original,
    X_test,
    y_test,
    'ORIGINAL DATASET - SIMPLE MODEL'
)

test_loss_simple_orig = results_simple_orig["loss"]
test_acc_simple_orig  = results_simple_orig["accuracy"]
f1_macro_simple_orig  = results_simple_orig["f1_macro"]
f1_weighted_simple_orig = results_simple_orig["f1_weighted"]

# Save model
os.makedirs('models', exist_ok=True)
simple_model_original.save('models/emnist_original_simple.keras')


In [ ]:
print("TRAINING ON ORIGINAL DATASET")

# Load data
def rotate(images):
    images = np.array([np.rot90(np.fliplr(img)) for img in images])
    return images

train_images = load_idx_images('data/original/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('data/original/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('data/original/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('data/original/emnist-byclass-test-labels-idx1-ubyte')

train_images = rotate(train_images)
test_images = rotate(test_images)

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile model
model_original = create_model()
model_original.summary()

initial_lr = 0.001
optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
model_original.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_original = model_original.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_original_model'),
    verbose=1
)

# Plot history
plot_history(history_original, 'Original Dataset')

# Evaluate
results_orig = evaluate_model(
    model_original,
    X_test,
    y_test,
    'ORIGINAL DATASET - STANDARD MODEL'
)

test_loss_orig = results_orig["loss"]
test_acc_orig  = results_orig["accuracy"]
f1_macro_orig  = results_orig["f1_macro"]
f1_weighted_orig = results_orig["f1_weighted"]

os.makedirs('models', exist_ok=True)
model_original.save('models/emnist_original.keras')

In [ ]:
print("TRAINING ON BALANCED DATASET - SIMPLE MODEL")

train_images = load_idx_images('data/balanced/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('data/balanced/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('data/balanced/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('data/balanced/emnist-byclass-test-labels-idx1-ubyte')

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile simple model
simple_model_balanced = create_simple_model()
simple_model_balanced.summary()

initial_lr = 0.001
optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
simple_model_balanced.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_simple_balanced = simple_model_balanced.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_balanced_simple_model'),
    verbose=1
)

# Plot history
plot_history(history_simple_balanced, 'Balanced Dataset - Simple Model')

# Evaluate
results_simple_balanced = evaluate_model(
    simple_model_balanced,
    X_test,
    y_test,
    'BALANCED DATASET - SIMPLE MODEL'
)

test_loss_simple_balanced = results_simple_balanced["loss"]
test_acc_simple_balanced  = results_simple_balanced["accuracy"]
f1_macro_simple_balanced  = results_simple_balanced["f1_macro"]
f1_weighted_simple_balanced = results_simple_balanced["f1_weighted"]

# Save model
os.makedirs('models', exist_ok=True)
simple_model_original.save('models/emnist_balanced_simple.keras')


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import os
from sklearn.metrics import f1_score

class EMNISTDataGenerator(keras.utils.Sequence):
    def __init__(self, images_file, labels_file, batch_size=128,
                 num_classes=62, shuffle=True, normalize=True):
        self.images_file = images_file
        self.labels_file = labels_file
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.shuffle = shuffle
        self.normalize = normalize

        with open(images_file, 'rb') as f:
            magic = int.from_bytes(f.read(4), 'big')
            self.n_samples = int.from_bytes(f.read(4), 'big')
            self.rows = int.from_bytes(f.read(4), 'big')
            self.cols = int.from_bytes(f.read(4), 'big')
            self.image_offset = f.tell()

        with open(labels_file, 'rb') as f:
            f.read(8)
            self.label_offset = f.tell()

        self.indices = np.arange(self.n_samples)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(self.n_samples / self.batch_size))

    def __getitem__(self, index):
        start_idx = index * self.batch_size
        end_idx = min((index + 1) * self.batch_size, self.n_samples)
        batch_indices = self.indices[start_idx:end_idx]

        batch_images = self._load_images(batch_indices)
        batch_labels = self._load_labels(batch_indices)

        X = batch_images.astype('float32')
        if self.normalize:
            X = X / 255.0
        X = X.reshape(-1, 28, 28, 1)

        y = keras.utils.to_categorical(batch_labels, self.num_classes)

        return X, y

    def _load_images(self, indices):
        batch_size = len(indices)
        images = np.zeros((batch_size, self.rows, self.cols), dtype=np.uint8)

        with open(self.images_file, 'rb') as f:
            for i, idx in enumerate(indices):
                offset = self.image_offset + idx * self.rows * self.cols
                f.seek(offset)
                img_data = f.read(self.rows * self.cols)
                images[i] = np.frombuffer(img_data, dtype=np.uint8).reshape(self.rows, self.cols)

        return images

    def _load_labels(self, indices):
        batch_size = len(indices)
        labels = np.zeros(batch_size, dtype=np.uint8)

        with open(self.labels_file, 'rb') as f:
            for i, idx in enumerate(indices):
                offset = self.label_offset + idx
                f.seek(offset)
                labels[i] = int.from_bytes(f.read(1), 'big')

        return labels

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

print("TRAINING ON AUGMENTED DATASET - SIMPLE MODEL (MEMORY EFFICIENT)")

train_generator = EMNISTDataGenerator(
    'data/augmented/emnist-byclass-train-images-idx3-ubyte',
    'data/augmented/emnist-byclass-train-labels-idx1-ubyte',
    batch_size=128,
    num_classes=62,
    shuffle=True
)

test_generator = EMNISTDataGenerator(
    'data/augmented/emnist-byclass-test-images-idx3-ubyte',
    'data/augmented/emnist-byclass-test-labels-idx1-ubyte',
    batch_size=128,
    num_classes=62,
    shuffle=False
)

print(f"Train: {train_generator.n_samples} images ({len(train_generator)} batches)")
print(f"Test: {test_generator.n_samples} images ({len(test_generator)} batches)\n")

# Create model
simple_model_augmented = create_simple_model()
simple_model_augmented.summary()

initial_lr = 0.001
optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
simple_model_augmented.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_simple_augmented = simple_model_augmented.fit(
    train_generator,
    epochs=50,
    validation_data=test_generator,
    callbacks=get_callbacks('emnist_augmented_simple_model'),
    verbose=1
)

# Plot history
plot_history(history_simple_augmented, 'Augmented Dataset - Simple Model')

# Evaluate 
results_simple_augmented = evaluate_model_generator(
    simple_model_augmented,
    test_generator,
    'AUGMENTED DATASET - SIMPLE MODEL'
)

test_loss_simple_augmented = results_simple_augmented["loss"]
test_acc_simple_augmented = results_simple_augmented["accuracy"]
f1_macro_simple_augmented = results_simple_augmented["f1_macro"]
f1_weighted_simple_augmented = results_simple_augmented["f1_weighted"]

# Save model
os.makedirs('models', exist_ok=True)
simple_model_augmented.save('models/emnist_augmented_simple.keras')


In [ ]:
print("TRAINING ON COMBINED DATASET - SIMPLE MODEL")

train_images = load_idx_images('data/combined/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('data/combined/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('data/combined/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('data/combined/emnist-byclass-test-labels-idx1-ubyte')

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile model
model_combined = create_model()
model_combined.summary()

initial_lr = 0.001
optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
model_combined.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_combined = model_combined.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_combined_model'),
    verbose=1
)

# Plot history
plot_history(history_combined, 'Combined Dataset - Simple Model')

# Evaluate
results_combined = evaluate_model(
    model_combined,
    X_test,
    y_test,
    'COMBINED DATASET - SIMPLE MODEL'
)

# Save model
os.makedirs('models', exist_ok=True)
model_combined.save('models/emnist_combined.keras')


In [ ]:
print("TRAINING FINAL MODEL")

train_images = load_idx_images('data/combined/emnist-byclass-train-images-idx3-ubyte')
train_labels = load_idx_labels('data/combined/emnist-byclass-train-labels-idx1-ubyte')
test_images = load_idx_images('data/combined/emnist-byclass-test-images-idx3-ubyte')
test_labels = load_idx_labels('data/combined/emnist-byclass-test-labels-idx1-ubyte')

print(f"Train: {len(train_images)} images")
print(f"Test: {len(test_images)} images\n")

# Prepare data
X_train, y_train = prepare_data(train_images, train_labels, num_classes=62)
X_test, y_test   = prepare_data(test_images, test_labels, num_classes=62)

# Create and compile simple model
model_final = create_simple_model()
model_final.summary()

initial_lr = 0.001
optimizer = keras.optimizers.Adam(learning_rate=initial_lr)
model_final.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_final = model_final.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    callbacks=get_callbacks('emnist_final_model'),
    verbose=1
)

# Plot history
plot_history(history_final, 'Final Dataset')

# Evaluate
results_final = evaluate_model(
    model_final,
    X_test,
    y_test,
    'FINAL DATASET'
)

# Save model
os.makedirs('models', exist_ok=True)
model_final.save('models/emnist_final.keras')
